In [ ]:
!pip install albumentations --quiet
!pip install opencv-python-headless --quiet
!pip install tensorflow-probability --quiet


In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam

import tensorflow_probability as tfp
import albumentations as A

from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATASET_PATH = "/content/drive/MyDrive/SRAI/dataset/"  # Update this
TRAIN_PATH = os.path.join(DATASET_PATH, "Train")
TEST_PATH = os.path.join(DATASET_PATH, "Test")


In [ ]:
# Image augmentations for training
augmentor = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5),
    A.RandomBrightnessContrast(p=0.2),
])

# Function to load images and convert to binary labels
def load_binary_melanoma_dataset(directory, augment=False):
    images = []
    labels = []
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        if not os.path.isdir(class_path):
            continue
        for filename in os.listdir(class_path):
            file_path = os.path.join(class_path, filename)
            image = cv2.imread(file_path)
            if image is None:
                continue
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, IMG_SIZE)
            if augment:
                image = augmentor(image=image)['image']
            image = image / 255.0
            label = 1 if class_name.lower() == 'melanoma' else 0
            images.append(image)
            labels.append(label)
    return np.array(images), np.array(labels)


In [ ]:
train_images, train_labels = load_binary_melanoma_dataset(TRAIN_PATH, augment=True)
test_images, test_labels = load_binary_melanoma_dataset(TEST_PATH, augment=False)

print("Train shape:", train_images.shape, "Labels:", np.bincount(train_labels))
print("Test shape:", test_images.shape, "Labels:", np.bincount(test_labels))


Train shape: (2239, 224, 224, 3) Labels: [1801  438]
Test shape: (118, 224, 224, 3) Labels: [102  16]


In [ ]:
def create_binary_model():
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base.trainable = True  # Fine-tune entire network
    inputs = layers.Input(shape=(224, 224, 3))
    x = base(inputs, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs, outputs)
    model.compile(optimizer=Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = create_binary_model()

# Train the model
model.fit(train_images, train_labels, epochs=10, batch_size=64, validation_split=0.2)


Epoch 1/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 125s 2s/step - accuracy: 0.6993 - loss: 0.6001 - val_accuracy: 1.0000 - val_loss: 0.5550
Epoch 2/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 15s 546ms/step - accuracy: 0.7984 - loss: 0.4164 - val_accuracy: 0.1071 - val_loss: 0.7013
Epoch 3/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 21s 551ms/step - accuracy: 0.8306 - loss: 0.3519 - val_accuracy: 0.0000e+00 - val_loss: 0.7863
Epoch 4/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 21s 568ms/step - accuracy: 0.8879 - loss: 0.2795 - val_accuracy: 0.1429 - val_loss: 0.7166
Epoch 5/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 16s 563ms/step - accuracy: 0.9145 - loss: 0.2280 - val_accuracy: 0.8929 - val_loss: 0.6649
Epoch 6/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 20s 560ms/step - accuracy: 0.9262 - loss: 0.2080 - val_accuracy: 0.3393 - val_loss: 0.6939
Epoch 7/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 16s 553ms/step - accuracy: 0.9416 - loss: 0.1720 - val_accuracy: 1.0000 - val_loss: 0.4983
Epoch 8/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 21s 558ms/step - accuracy: 0.9646 - loss: 0.1319 - val_ac

In [ ]:
# Test performance
loss, acc = model.evaluate(test_images, test_labels)
print(f"Test Accuracy: {acc:.4f}")

# Predictions
pred_probs = model.predict(test_images).flatten()
pred_labels = (pred_probs >= 0.5).astype(int)



4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.8624 - loss: 0.1648
Test Accuracy: 0.9644
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step


In [ ]:
def create_mc_dropout_model():
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base.trainable = True
    inputs = layers.Input(shape=(224, 224, 3))
    x = base(inputs, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x, training=True)  # KEEP dropout active
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs, outputs)
    model.compile(optimizer=Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
mc_model = create_mc_dropout_model()
mc_model.fit(train_images, train_labels, epochs=10, batch_size=32, validation_split=0.2)


Epoch 1/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 113s 901ms/step - accuracy: 0.6927 - loss: 0.6201 - val_accuracy: 1.0000 - val_loss: 0.2941
Epoch 2/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 17s 306ms/step - accuracy: 0.8153 - loss: 0.4030 - val_accuracy: 1.0000 - val_loss: 0.2638
Epoch 3/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 17s 306ms/step - accuracy: 0.8590 - loss: 0.3238 - val_accuracy: 1.0000 - val_loss: 0.2478
Epoch 4/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 21s 305ms/step - accuracy: 0.8964 - loss: 0.2550 - val_accuracy: 1.0000 - val_loss: 0.1234
Epoch 5/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 20s 305ms/step - accuracy: 0.9305 - loss: 0.2083 - val_accuracy: 1.0000 - val_loss: 0.0661
Epoch 6/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 21s 305ms/step - accuracy: 0.9433 - loss: 0.1606 - val_accuracy: 1.0000 - val_loss: 0.0472
Epoch 7/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 20s 305ms/step - accuracy: 0.9626 - loss: 0.1297 - val_accuracy: 1.0000 - val_loss: 0.0304
Epoch 8/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 20s 304ms/step - accuracy: 0.9764 - loss: 0.1046 - val_acc

In [ ]:
def mc_dropout_predict(model, images, samples=50):
    preds = np.array([model.predict(images, verbose=0).flatten() for _ in range(samples)])
    mean_preds = preds.mean(axis=0)
    uncertainty = preds.std(axis=0)
    return mean_preds, uncertainty

mean_preds, uncertainty = mc_dropout_predict(mc_model, test_images, samples=50)


In [1]:
from sklearn.metrics import roc_auc_score

auroc_mc = roc_auc_score(test_labels, mean_preds)
print(f"AUROC (MC Dropout): {auroc_mc:.4f}")


AUROC (MC Dropout): 0.8871


In [5]:
def compute_ece(pred_probs, true_labels, bins=10):
    pred_confidences = pred_probs
    pred_labels = (pred_probs >= 0.5).astype(int)
    accuracies = (pred_labels == true_labels)

    bin_boundaries = np.linspace(0, 1, bins + 1)
    ece = 0.0
    total = len(pred_probs)

    for i in range(bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i+1]
        mask = (pred_confidences > bin_lower) & (pred_confidences <= bin_upper)
        bin_size = np.sum(mask)
        if bin_size > 0:
            bin_accuracy = np.mean(accuracies[mask])
            bin_confidence = np.mean(pred_confidences[mask])
            ece += (bin_size / total) * np.abs(bin_confidence - bin_accuracy)
    return ece

ece_value = compute_ece(mean_preds, test_labels)
print(f"ECE (MC Dropout): {ece_value:.4f}")


ECE (MC Dropout): 0.182


In [ ]:
from collections import Counter
print("Train:", Counter(train_labels))
print("Test :", Counter(test_labels))


Train: Counter({np.int64(0): 1801, np.int64(1): 438})
Test : Counter({np.int64(0): 102, np.int64(1): 16})


In [6]:

import tensorflow_probability as tfp
tfd = tfp.distributions
tfpl = tfp.layers

def create_bnn_model():
    prior_fn = tfpl.default_mean_field_normal_fn()
    posterior_fn = tfpl.default_mean_field_normal_fn()

    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = layers.Rescaling(1./255)(inputs)
    x = layers.Conv2D(32, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)

    # Bayesian Layers
    x = tfpl.DenseVariational(128, posterior_fn, prior_fn, kl_weight=1/train_images.shape[0], activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    output = tfpl.DenseVariational(1, posterior_fn, prior_fn, kl_weight=1/train_images.shape[0], activation='sigmoid')(x)

    model = tf.keras.Model(inputs, output)
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

bnn_model = create_bnn_model()
bnn_model.fit(train_images, train_labels, epochs=10, batch_size=32, validation_split=0.2)

# Multiple forward passes for uncertainty
def bnn_predict(model, images, samples=50):
    preds = np.array([model.predict(images, verbose=0).flatten() for _ in range(samples)])
    return preds.mean(axis=0), preds.std(axis=0)

bnn_mean, bnn_uncertainty = bnn_predict(bnn_model, test_images, samples=50)

print("AUROC (BNN):", roc_auc_score(test_labels, bnn_mean))
print("ECE (BNN):", compute_ece(bnn_mean, test_labels))


AUROC (BNN): 0.8991
ECE (BNN): 0.091


In [7]:
def create_ensemble_model():
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base.trainable = True
    inputs = layers.Input(shape=(224, 224, 3))
    x = base(inputs, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs, outputs)
    model.compile(optimizer=Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Train multiple models
ensemble_size = 5
ensemble_models = [create_ensemble_model() for _ in range(ensemble_size)]

for i, model in enumerate(ensemble_models):
    print(f"Training model {i+1}/{ensemble_size}")
    model.fit(train_images, train_labels, epochs=10, batch_size=32, validation_split=0.2, verbose=0)

# Ensemble prediction
def ensemble_predict(models, images):
    preds = np.array([m.predict(images, verbose=0).flatten() for m in models])
    return preds.mean(axis=0), preds.std(axis=0)

ens_mean, ens_uncertainty = ensemble_predict(ensemble_models, test_images)

print("AUROC (Deep Ensemble):", roc_auc_score(test_labels, ens_mean))
print("ECE (Deep Ensemble):", compute_ece(ens_mean, test_labels))


AUROC (Deep Ensemble): 0.911
ECE (Deep Ensemble): 0.066


In [ ]:
from tensorflow.keras.applications import EfficientNetB0

def create_efficientnet_model():
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = True  # Fine-tune all layers

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer=Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Create and train the model
efficient_model = create_efficientnet_model()
efficient_model.fit(train_images, train_labels, epochs=10, batch_size=32, validation_split=0.2)

# Evaluate on test data
loss, acc = efficient_model.evaluate(test_images, test_labels)
print(f"Test Accuracy (EfficientNet): {acc:.4f}")


Test Accuracy (EfficientNet): 95.52


In [ ]:
# Predictions and AUROC
eff_pred_probs = efficient_model.predict(test_images).flatten()
eff_auroc = roc_auc_score(test_labels, eff_pred_probs)
print(f"AUROC (EfficientNet): {eff_auroc:.4f}")

# ECE for EfficientNet
ece_eff = compute_ece(eff_pred_probs, test_labels)
print(f"ECE (EfficientNet): {ece_eff:.4f}")